# Xin Ying's Deep Learning Pipeline: PyTorch TabularNN with Entity Embeddings
**Course:** IT2226 - AI Innovation and Deep Learning Project  
**Target:** Predict 30-day repeat purchase in October 2024 (Class 1 = 14.57% positive)  
**Validation:** 80/20 Holdout Split & 5-Fold Stratified Cross-Validation (`seed=42`)

### Key Highlights:
1. **Categorical Entity Embeddings:** Dense vector representations for `country_code`, `customer_tier`, and `preferred_channel`.
2. **Continuous Feature Normalization:** Batch normalization and standard scaling applied strictly to training folds to prevent leakage.
3. **Class Imbalance Handling:** Positive class weighting (`pos_weight = 5.863`) in BCE loss.
4. **Benchmarked:** Compared against the team LightGBM baseline.

In [ ]:
# 1. Setup & Environment
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report, f1_score, roc_curve, precision_recall_curve
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Set seeds globally for 100% deterministic reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch Version: {torch.__version__} | Device: {DEVICE}')

In [ ]:
# 2. Load Customer 360 Feature Matrix
FEATURE_PATH = 'task_outputs/model_feature_matrix.csv'
if not os.path.exists(FEATURE_PATH):
    FEATURE_PATH = '../../task_outputs/model_feature_matrix.csv'

df = pd.read_csv(FEATURE_PATH)
print(f'Total customer records: {len(df):,}')

CAT_COLS = ['country_code', 'customer_tier', 'preferred_channel']
DROP_COLS = ['customer_id', 'target', 'is_test']
NUM_COLS = [c for c in df.columns if c not in CAT_COLS and c not in DROP_COLS]

train_df = df[df['is_test'] == 0].copy().reset_index(drop=True)
test_df = df[df['is_test'] == 1].copy().reset_index(drop=True)

print(f'Train samples: {len(train_df):,} | Test samples: {len(test_df):,}')
print(f'Categorical columns ({len(CAT_COLS)}): {CAT_COLS}')
print(f'Continuous columns ({len(NUM_COLS)}): {len(NUM_COLS)} features')

In [ ]:
# 3. Categorical Encodings & Median Imputation
cat_dims = []
cat_encoders = {}

for c in CAT_COLS:
    train_df[c] = train_df[c].fillna('Unknown').astype(str)
    test_df[c] = test_df[c].fillna('Unknown').astype(str)
    le = LabelEncoder()
    train_df[c] = le.fit_transform(train_df[c])
    test_df[c] = le.transform(test_df[c])
    cat_encoders[c] = le
    n_classes = len(le.classes_)
    emb_dim = max(4, (n_classes + 1) // 2)
    cat_dims.append((n_classes, emb_dim))

for col in NUM_COLS:
    med = train_df[col].median()
    train_df[col] = train_df[col].fillna(med)
    test_df[col] = test_df[col].fillna(med)

y = train_df['target'].astype(int).values
pos_weight_val = (len(y) - y.sum()) / y.sum()
print(f'Pos weight: {pos_weight_val:.4f}')

In [ ]:
# 4. PyTorch Dataset and TabularNN Architecture
class TabularDataset(Dataset):
    def __init__(self, cat_data, num_data, targets=None):
        self.cat_data = torch.tensor(cat_data, dtype=torch.long)
        self.num_data = torch.tensor(num_data, dtype=torch.float32)
        self.targets = torch.tensor(targets, dtype=torch.float32) if targets is not None else None

    def __len__(self):
        return len(self.cat_data)

    def __getitem__(self, idx):
        if self.targets is not None:
            return self.cat_data[idx], self.num_data[idx], self.targets[idx]
        return self.cat_data[idx], self.num_data[idx]

class TabularNN(nn.Module):
    def __init__(self, emb_dims, num_numerical, hidden_dims=[128, 64, 32], dropout_p=0.25):
        super().__init__()
        self.embeddings = nn.ModuleList([nn.Embedding(num_c, dim) for num_c, dim in emb_dims])
        total_emb_dim = sum(dim for _, dim in emb_dims)
        self.num_bn = nn.BatchNorm1d(num_numerical)
        
        layers = []
        in_dim = total_emb_dim + num_numerical
        for h_dim in hidden_dims:
            layers.append(nn.Linear(in_dim, h_dim))
            layers.append(nn.BatchNorm1d(h_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_p))
            in_dim = h_dim
        layers.append(nn.Linear(in_dim, 1))
        self.mlp = nn.Sequential(*layers)

    def forward(self, x_cat, x_num):
        emb_outs = [emb_layer(x_cat[:, i]) for i, emb_layer in enumerate(self.embeddings)]
        x_emb = torch.cat(emb_outs, dim=1)
        x_num_norm = self.num_bn(x_num)
        x = torch.cat([x_emb, x_num_norm], dim=1)
        return self.mlp(x).squeeze(1)

print('TabularNN Model Class Compiled Successfully.')

In [ ]:
# 5. Train on 80%, Test on 20% Holdout Split
X_cat_all = train_df[CAT_COLS].values
X_num_all = train_df[NUM_COLS].values

X_cat_tr, X_cat_va, X_num_tr_raw, X_num_va_raw, y_tr, y_va = train_test_split(
    X_cat_all, X_num_all, y, test_size=0.20, stratify=y, random_state=SEED
)

# Strictly scale using training set to prevent data leakage
scaler = StandardScaler()
X_num_tr = scaler.fit_transform(X_num_tr_raw)
X_num_va = scaler.transform(X_num_va_raw)

train_loader = DataLoader(TabularDataset(X_cat_tr, X_num_tr, y_tr), batch_size=512, shuffle=True)
val_loader = DataLoader(TabularDataset(X_cat_va, X_num_va, y_va), batch_size=512, shuffle=False)

model = TabularNN(cat_dims, len(NUM_COLS)).to(DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight_val], device=DEVICE))
optimizer = torch.optim.AdamW(model.parameters(), lr=0.002, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=15)

for epoch in range(1, 13):
    model.train()
    for bc, bn, bt in train_loader:
        bc, bn, bt = bc.to(DEVICE), bn.to(DEVICE), bt.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(bc, bn), bt)
        loss.backward()
        optimizer.step()
    scheduler.step()

model.eval()
val_preds = []
with torch.no_grad():
    for bc, bn, _ in val_loader:
        bc, bn = bc.to(DEVICE), bn.to(DEVICE)
        val_preds.extend(torch.sigmoid(model(bc, bn)).cpu().numpy())

val_probs = np.array(val_preds)
val_auc = roc_auc_score(y_va, val_probs)
val_pr_auc = average_precision_score(y_va, val_probs)

best_th, best_f1 = 0.5, 0.0
for th in np.arange(0.2, 0.8, 0.02):
    f1 = f1_score(y_va, (val_probs >= th).astype(int))
    if f1 > best_f1:
        best_f1, best_th = f1, th

rec = np.sum((val_probs >= best_th) & (y_va == 1)) / np.sum(y_va == 1)

print('='*55)
print(f'80/20 HOLDOUT EVALUATION (PyTorch TabularDL):')
print(f'ROC-AUC:   {val_auc:.4f}')
print(f'PR-AUC:    {val_pr_auc:.4f}')
print(f'Best F1:   {best_f1:.4f} (at threshold {best_th:.2f})')
print(f'Recall:    {rec:.4f} ({rec*100:.2f}%)')
print('='*55)